In [ ]:
import os
import time
import re
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import BertTokenizer, BertModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Set up global parameters
DATA_DIRECTORY = "./ids_data" 
TARGET_SAMPLE_SIZE = 100000
TESTING_SLICE_LIMIT = 15000  # Safe boundary for CPU execution

print("Environment libraries successfully loaded.")

In [ ]:
all_files = os.listdir(DATA_DIRECTORY)
parquet_files = [os.path.join(DATA_DIRECTORY, f) for f in all_files if f.endswith('.parquet')]
csv_files = [os.path.join(DATA_DIRECTORY, f) for f in all_files if f.endswith('.csv')]

if not parquet_files and not csv_files:
    raise FileNotFoundError(f"No usable data logs discovered in '{DATA_DIRECTORY}'. Please verify your file placement.")

collected_frames = []

# 1. Process 2018 Parquet Files
for file_path in parquet_files:
    print(f"Ingesting 2018 Parquet file: {os.path.basename(file_path)}")
    df_chunk = pd.read_parquet(file_path, engine='pyarrow')
    df_chunk.columns = df_chunk.columns.str.strip()
    
    if 'Label' in df_chunk.columns:
        df_chunk = df_chunk.dropna(subset=['Label'])
        collected_frames.append(df_chunk)

# 2. Process 2017 CSV Files
for file_path in csv_files:
    print(f"Ingesting 2017 CSV file: {os.path.basename(file_path)}")
    for chunk in pd.read_csv(file_path, chunksize=50000, low_memory=False):
        chunk.columns = chunk.columns.str.strip()
        
        if 'Label' in chunk.columns:
            chunk['Label'] = chunk['Label'].str.strip()
            chunk['Label'] = chunk['Label'].replace({
                'BENIGN': 'Benign',
                'DDoS': 'Attack',
                'DoS Hulk': 'Attack',
                'DoS GoldenEye': 'Attack',
                'DoS slowloris': 'Attack'
            })
            chunk = chunk.dropna(subset=['Label'])
            collected_frames.append(chunk)
            
        current_pool = sum(len(df) for df in collected_frames)
        if current_pool >= TARGET_SAMPLE_SIZE * 3:
            break

master_expanded_df = pd.concat(collected_frames)
print(f"\nTotal multi-generation dataset successfully loaded: {len(master_expanded_df)} rows.")

sampling_ratio = TARGET_SAMPLE_SIZE / len(master_expanded_df)
stratified_subset, _ = train_test_split(
    master_expanded_df, 
    train_size=sampling_ratio, 
    stratify=master_expanded_df['Label'], 
    random_state=42
)

print(f"\nStratified sample allocation complete. Total target size: {len(stratified_subset)} rows.")
print("Updated Training Profile Class Distribution:")
print(stratified_subset['Label'].value_counts())

In [ ]:
print("Converting tabular network logs into natural linguistic sentences...")

def build_semantic_log_string(row):
    """
    Transforms rigid CSV feature vectors into standardized narrative structures 
    so the BERT attention layers can interpret structural dependencies accurately.
    """
    dest_port = str(row.get('Dst Port', '0')).split('.')[0]
    protocol = str(row.get('Protocol', '0')).split('.')[0]
    duration = str(row.get('Flow Duration', '0'))
    fwd_pkts = str(row.get('Tot Fwd Pkts', '0'))
    bwd_pkts = str(row.get('Tot Bwd Pkts', '0'))
    
    log_line = (f"Session target port: {dest_port} using transmission protocol: {protocol}. "
                f"Active connection duration recorded at {duration} microseconds. "
                f"Metrics outbound pack count: {fwd_pkts} and inbound payload pack count: {bwd_pkts}.")
    return log_line

# Apply sentence transformation across the subset rows
stratified_subset['Log_Syntax_String'] = stratified_subset.apply(build_semantic_log_string, axis=1)

# Convert arrays into plain python lists for tokenizer ingestion
X_text_corpus = stratified_subset['Log_Syntax_String'].tolist()
y_targets = np.where(stratified_subset['Label'] == 'Benign', 0, 1)

print("Sample transformed log format:")
print(X_text_corpus[0])

In [ ]:
print("Configuring Hugging Face BERT Transformer infrastructure...")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_engine = BertModel.from_pretrained('bert-base-uncased')

# Force target compute engine to CPU or GPU depending on hardware status
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Targeting compute engine: {device}")
bert_engine.to(device)
bert_engine.eval()

def generate_bert_feature_matrix(text_list, batch_size=64):
    matrix_collector = []
    total_items = len(text_list)
    
    for idx in range(0, total_items, batch_size):
        batch_slice = text_list[idx : idx + batch_size]
        
        # Tokenize batch strings
        tokens = tokenizer(batch_slice, padding=True, truncation=True, max_length=64, return_tensors="pt")
        tokens = {k: v.to(device) for k, v in tokens.items()}
        
        with torch.no_grad():
            outputs = bert_engine(**tokens)
            
        # Extract the high-dimensional hidden state vector from the [CLS] anchor token position
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        matrix_collector.append(cls_embeddings)
        
        if (idx + batch_size) % (batch_size * 5) == 0 or (idx + batch_size) >= total_items:
            processed = min(idx + batch_size, total_items)
            print(f"Embedded data logs tracked: {processed}/{total_items}")
            
    return np.vstack(matrix_collector)

# Isolate your execution slice boundary
X_truncated_text = X_text_corpus[:TESTING_SLICE_LIMIT]
y_truncated_labels = y_targets[:TESTING_SLICE_LIMIT]

print(f"Slicing initial text profiles down to first {TESTING_SLICE_LIMIT} logs for validation...")
embedding_timer_start = time.time()
X_embedding_features = generate_bert_feature_matrix(X_truncated_text)
print(f"Matrix feature extraction completed in: {time.time() - embedding_timer_start:.2f} seconds.")



In [ ]:
# Standard 70/30 training vs verification split logic
X_train, X_test, y_train, y_test = train_test_split(
    X_embedding_features, y_truncated_labels, test_size=0.3, random_state=42
)

print(f"Training shape matrix: {X_train.shape} | Evaluation matrix shape: {X_test.shape}")
print("Configuring optimized Random Forest ensemble classifier...")

# Maximize core allocation via n_jobs=-1 for extreme processing execution
random_forest = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
random_forest.fit(X_train, y_train)

print("Running testing inferences...")
inference_timer_start = time.time()
test_predictions = random_forest.predict(X_test)
inference_timer_end = time.time()

total_test_instances = len(X_test)
total_latency_seconds = inference_timer_end - inference_timer_start

# Print complete CSCI 350 Performance Metrics Report

print("CSCI 350 EXPERIMENTAL EVALUATION REPORT")

print(f"Evaluated Test Packets   : {total_test_instances}")
print(f"Total Processing Latency : {total_latency_seconds:.5f} seconds")
print(f"Average Per-Packet Speed : {(total_latency_seconds / total_test_instances) * 1000:.4f} milliseconds/packet")
print(f"Overall Model Accuracy   : {accuracy_score(y_test, test_predictions) * 100:.2f}%")

print("Detailed Model Classification Performance Profile:")
print(classification_report(y_test, test_predictions, target_names=["Benign (0)", "Attack Anomaly (1)"]))






In [ ]:
import os
import time
import psutil
import numpy as np

def evaluate_system_resource_costs(execution_function, *args, **kwargs):
    """
    Executes a target machine learning function while continuously 
    monitoring system-level memory overhead and hardware utilization.
    """
    # Target the current running python process id
    process = psutil.Process(os.getpid())
    
    # Capture resource baseline values before execution
    baseline_memory = process.memory_info().rss / (1024 * 1024) # Convert bytes to Megabytes
    cpu_cores_count = psutil.cpu_count()
    
    print("=" * 60)
    print("SYSTEM RESOURCE UTILIZATION MONITOR ACTIVE")
    print("=" * 60)
    print(f"Hardware Allocation Platform : Single Node CPU Engine")
    print(f"Available Logical CPU Cores : {cpu_cores_count} cores")
    print(f"Baseline Process RAM Space   : {baseline_memory:.2f} MB")
    print("-" * 60)
    
    # Initialize utilization sampling arrays
    cpu_samples = []
    
    # Start execution timer
    start_time = time.time()
    
    # Run the model logic while tracking performance metrics
    # (Using a small interval loop sample to avoid benchmarking lag)
    psutil.cpu_percent(interval=None) # Reset CPU counter
    
    result = execution_function(*args, **kwargs)
    
    # Capture peak execution metrics
    end_time = time.time()
    peak_memory = process.memory_info().rss / (1024 * 1024)
    avg_cpu_utilization = psutil.cpu_percent(interval=None)
    
    # Calculate delta variations
    total_elapsed_time = end_time - start_time
    net_memory_overhead = peak_memory - baseline_memory
    
    print("\n" + "-" * 60)
    print("RESOURCE METRICS EVALUATION SUMMARY")
    print("-" * 60)
    print(f"Total Execution Runtime     : {total_elapsed_time:.4f} seconds")
    print(f"Peak Operational RAM Space  : {peak_memory:.2f} MB")
    print(f"Net Added Memory Overhead   : {net_memory_overhead:.2f} MB")
    print(f"Average CPU Load Factor     : {avg_cpu_utilization:.1f}% Utilization Across Cores")
    print("=" * 60)
    
    return result, {
        "runtime": total_elapsed_time,
        "peak_ram": peak_memory,
        "net_ram": net_memory_overhead,
        "cpu_load": avg_cpu_utilization
    }

# HOW TO RUN IT INSIDE YOUR NOTEBOOK
# You wrap your existing embedding extraction function or random forest training step inside it:
# text_embeddings, metrics = evaluate_system_resource_costs(generate_bert_feature_matrix, X_truncated_text)

In [ ]:
# Compute confusion matrix array metrics
cm = confusion_matrix(y_test, test_predictions)

# Create structural diagram layout
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=["Benign (0)", "Attack Anomaly (1)"],
            yticklabels=["Benign (0)", "Attack Anomaly (1)"])

plt.title("Hybrid BERT-Random Forest DDoS Detection Performance", fontsize=12, pad=15)
plt.ylabel("Actual Baseline Classification", fontsize=10)
plt.xlabel("Predicted Model Classification", fontsize=10)
plt.tight_layout()

# Save the matrix image asset automatically for your proposal document insertion
plt.savefig("ddos_confusion_matrix.png", dpi=300)
plt.show()